In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
df = pd.read_csv('used_cars.csv')

In [5]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [6]:
df.shape

(4009, 12)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   brand         4009 non-null   str  
 1   model         4009 non-null   str  
 2   model_year    4009 non-null   int64
 3   milage        4009 non-null   str  
 4   fuel_type     3839 non-null   str  
 5   engine        4009 non-null   str  
 6   transmission  4009 non-null   str  
 7   ext_col       4009 non-null   str  
 8   int_col       4009 non-null   str  
 9   accident      3896 non-null   str  
 10  clean_title   3413 non-null   str  
 11  price         4009 non-null   str  
dtypes: int64(1), str(11)
memory usage: 376.0 KB


In [8]:
df.isnull().sum()

brand             0
model             0
model_year        0
milage            0
fuel_type       170
engine            0
transmission      0
ext_col           0
int_col           0
accident        113
clean_title     596
price             0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df.describe()

,model_year
count,4009.000000
mean,2015.515590
std,6.104816
min,1974.000000
25%,2012.000000
50%,2017.000000
75%,2020.000000
max,2024.000000


In [11]:
df.nunique()

brand             57
model           1898
model_year        34
milage          2818
fuel_type          7
engine          1146
transmission      62
ext_col          319
int_col          156
accident           2
clean_title        1
price           1569
dtype: int64

inspect the categories

In [12]:
df['brand'].value_counts()

brand
Ford             386
BMW              375
Mercedes-Benz    315
Chevrolet        292
Porsche          201
Audi             200
Toyota           199
Lexus            163
Jeep             143
Land             130
Nissan           116
Cadillac         107
GMC               91
RAM               91
Dodge             90
Tesla             87
Kia               76
Hyundai           72
Acura             64
Subaru            64
Mazda             64
Honda             63
INFINITI          59
Volkswagen        59
Lincoln           52
Jaguar            47
Volvo             38
Maserati          34
Bentley           33
MINI              33
Buick             30
Chrysler          28
Lamborghini       26
Genesis           20
Mitsubishi        20
Alfa              19
Rivian            17
Hummer            16
Pontiac           15
Ferrari           12
Rolls-Royce       11
Aston              9
Scion              6
McLaren            6
Saturn             5
FIAT               5
Lotus              4
Lucid  

In [13]:
df['fuel_type'].value_counts(dropna=False)

fuel_type
Gasoline          3309
Hybrid             194
NaN                170
E85 Flex Fuel      139
Diesel             116
–                   45
Plug-In Hybrid      34
not supported        2
Name: count, dtype: int64

In [14]:
df['transmission'].value_counts().head(10)

transmission
A/T                               1037
8-Speed A/T                        406
Transmission w/Dual Shift Mode     398
6-Speed A/T                        362
6-Speed M/T                        248
Automatic                          237
7-Speed A/T                        209
8-Speed Automatic                  176
10-Speed A/T                       119
5-Speed A/T                         86
Name: count, dtype: int64

In [15]:
df['accident'].value_counts(dropna=False)

accident
None reported                             2910
At least 1 accident or damage reported     986
NaN                                        113
Name: count, dtype: int64

In [16]:
df['clean_title'].value_counts(dropna=False)

clean_title
Yes    3413
NaN     596
Name: count, dtype: int64

In [17]:
df["engine"].value_counts().head(30)

engine
2.0L I4 16V GDI DOHC Turbo                               52
355.0HP 5.3L 8 Cylinder Engine Gasoline Fuel             48
420.0HP 6.2L 8 Cylinder Engine Gasoline Fuel             47
–                                                        45
300.0HP 3.0L Straight 6 Cylinder Engine Gasoline Fuel    44
240.0HP 2.0L 4 Cylinder Engine Gasoline Fuel             42
285.0HP 3.6L V6 Cylinder Engine Gasoline Fuel            40
5.7L V8 16V MPFI OHV                                     29
340.0HP 3.0L V6 Cylinder Engine Gasoline Fuel            28
3.6L V6 24V MPFI DOHC                                    28
3.6L V6 24V GDI DOHC                                     28
268.0HP 3.5L V6 Cylinder Engine Gasoline Fuel            24
455.0HP 6.2L 8 Cylinder Engine Gasoline Fuel             24
302.0HP 3.5L V6 Cylinder Engine Gasoline Fuel            23
490.0HP 6.2L 8 Cylinder Engine Gasoline Fuel             22
445.0HP 4.4L 8 Cylinder Engine Gasoline Fuel             21
295.0HP 3.5L V6 Cylinder Engine G

In [18]:
df["model"].value_counts().head(20)

model
M3 Base                  30
F-150 XLT                24
Corvette Base            22
1500 Laramie             18
Model Y Long Range       17
Camaro 2SS               17
Wrangler Sport           17
Mustang GT Premium       16
911 Carrera              16
M4 Base                  15
911 Carrera S            14
Explorer XLT             14
F-250 Lariat             14
M5 Base                  13
E-Class E 350 4MATIC     13
F-150 Lariat             13
E-Class E 350            13
R1S Adventure Package    12
Land Cruiser Base        12
ES 350 Base              12
Name: count, dtype: int64

In [19]:
df.columns

Index(['brand', 'model', 'model_year', 'milage', 'fuel_type', 'engine',
       'transmission', 'ext_col', 'int_col', 'accident', 'clean_title',
       'price'],
      dtype='str')

# cleaning

In [20]:
df_clean = df.copy()

### drop clean_title and model columns
 - clean_title has only one actual value hence no variation
 - model had 1898 unique categories out of 4009 rows, would therefore overfit because some models may be only one or two 


In [21]:
df_clean = df_clean.drop(columns=["clean_title", "model"])

In [22]:
df_clean.head()

,brand,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,price
0,Ford,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,"$10,300"
1,Hyundai,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,"$38,005"
2,Lexus,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,"$54,598"
3,INFINITI,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,"$15,500"
4,Audi,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,"$34,999"


### convert price to numeric

In [23]:
df_clean['price'] = df_clean['price'].str.replace('$', '').str.replace(',', '').astype(float)

In [24]:
df_clean['price'].head()

0    10300.0
1    38005.0
2    54598.0
3    15500.0
4    34999.0
Name: price, dtype: float64

In [25]:
df_clean['price'].describe()

count    4.009000e+03
mean     4.455319e+04
std      7.871064e+04
min      2.000000e+03
25%      1.720000e+04
50%      3.100000e+04
75%      4.999000e+04
max      2.954083e+06
Name: price, dtype: float64

### convert mileage 

In [26]:
df_clean['milage'] = df_clean['milage'].str.replace(' mi.', '').str.replace(',', '').astype(float)

In [27]:
df_clean['milage'].head()

0    51000.0
1    34742.0
2    22372.0
3    88900.0
4     9835.0
Name: milage, dtype: float64

In [28]:
df_clean[["milage", "price"]].describe()

,milage,price
count,4009.000000,4.009000e+03
mean,64717.551010,4.455319e+04
std,52296.599459,7.871064e+04
min,100.000000,2.000000e+03
25%,23044.000000,1.720000e+04
50%,52775.000000,3.100000e+04
75%,94100.000000,4.999000e+04
max,405000.000000,2.954083e+06


In [29]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   brand         4009 non-null   str    
 1   model_year    4009 non-null   int64  
 2   milage        4009 non-null   float64
 3   fuel_type     3839 non-null   str    
 4   engine        4009 non-null   str    
 5   transmission  4009 non-null   str    
 6   ext_col       4009 non-null   str    
 7   int_col       4009 non-null   str    
 8   accident      3896 non-null   str    
 9   price         4009 non-null   float64
dtypes: float64(2), int64(1), str(7)
memory usage: 313.3 KB


### handle missing values

In [30]:
#check missing values
df_clean.isnull().sum()

brand             0
model_year        0
milage            0
fuel_type       170
engine            0
transmission      0
ext_col           0
int_col           0
accident        113
price             0
dtype: int64

In [31]:
#look at row with missing values
df_clean[df_clean["fuel_type"].isna()].head(15)

,brand,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,price
9,Tesla,2020,34000.0,NaN,534.0HP Electric Motor Electric Fuel System,A/T,Black,Black,None reported,69950.0
44,Lucid,2022,3552.0,NaN,536.0HP Electric Motor Electric Fuel System,1-Speed A/T,Red,Beige,None reported,119999.0
68,Lucid,2022,4900.0,NaN,536.0HP Electric Motor Electric Fuel System,1-Speed A/T,Red,Black,None reported,99000.0
92,Rivian,2023,2800.0,NaN,835.0HP Electric Motor Electric Fuel System,1-Speed A/T,White,Green,None reported,92000.0
122,Rivian,2023,2500.0,NaN,835.0HP Electric Motor Electric Fuel System,A/T,Green,White,None reported,94000.0
129,Lucid,2023,1300.0,NaN,620.0HP Electric Motor Electric Fuel System,A/T,Black,Gray,NaN,86900.0
155,Tesla,2022,13079.0,NaN,455.0HP Electric Motor Electric Fuel System,A/T,Black,White,None reported,47000.0
189,Tesla,2023,500.0,NaN,455.0HP Electric Motor Electric Fuel System,1-Speed A/T,Black,White,None reported,60000.0
225,Tesla,2023,8200.0,NaN,670.0HP Electric Motor Electric Fuel System,A/T,Black,Black,None reported,93999.0
236,Polestar,2021,12172.0,NaN,Electric,1-Speed Automatic,Thunder Gray,Charcoal,None reported,35999.0


In [32]:
df_clean[df_clean["accident"].isna()].head()

,brand,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,price
20,Genesis,2023,5400.0,Gasoline,375.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Green,Beige,NaN,60000.0
89,Subaru,2004,210703.0,Gasoline,165.0HP 2.5L 4 Cylinder Engine Gasoline Fuel,M/T,Green,Beige,NaN,2300.0
128,Audi,2015,98527.0,Gasoline,280.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Black,Black,NaN,9995.0
129,Lucid,2023,1300.0,NaN,620.0HP Electric Motor Electric Fuel System,A/T,Black,Gray,NaN,86900.0
131,Audi,2012,72922.0,Gasoline,265.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,6-Speed A/T,White,Black,NaN,25999.0


In [33]:
df_clean[df_clean["fuel_type"].isna()]["engine"].value_counts().head(20)

engine
Electric                                             17
835.0HP Electric Motor Electric Fuel System          16
425.0HP Electric Motor Electric Fuel System          16
455.0HP Electric Motor Electric Fuel System          12
518.0HP Electric Motor Electric Fuel System          11
271.0HP Electric Motor Electric Fuel System           9
Electric Motor Electric Fuel System                   9
670.0HP Electric Motor Electric Fuel System           7
170.0HP 0.65L Electric Motor Electric Fuel System     5
200.0HP Electric Motor Electric Fuel System           4
355.0HP Electric Motor Electric Fuel System           3
329.0HP Electric Motor Electric Fuel System           3
Dual Motor - Standard                                 3
201.0HP Electric Motor Electric Fuel System           3
320.0HP Electric Motor Electric Fuel System           3
295.0HP Electric Motor Electric Fuel System           3
480.0HP Electric Motor Electric Fuel System           3
534.0HP Electric Motor Electric Fuel Syst

this shows that the rows missing fuel type are electric

#### fill missing fuel types

In [34]:
df_clean["fuel_type"] = df_clean["fuel_type"].fillna("Electric")

In [35]:
df_clean["fuel_type"].value_counts(dropna=False)

fuel_type
Gasoline          3309
Hybrid             194
Electric           170
E85 Flex Fuel      139
Diesel             116
–                   45
Plug-In Hybrid      34
not supported        2
Name: count, dtype: int64

In [36]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   brand         4009 non-null   str    
 1   model_year    4009 non-null   int64  
 2   milage        4009 non-null   float64
 3   fuel_type     4009 non-null   str    
 4   engine        4009 non-null   str    
 5   transmission  4009 non-null   str    
 6   ext_col       4009 non-null   str    
 7   int_col       4009 non-null   str    
 8   accident      3896 non-null   str    
 9   price         4009 non-null   float64
dtypes: float64(2), int64(1), str(7)
memory usage: 313.3 KB


#### investigate accidents

In [37]:
df_clean[df_clean["accident"].isna()][["brand", "model_year", "milage", "accident", "price"]].head(20)

,brand,model_year,milage,accident,price
20,Genesis,2023,5400.0,NaN,60000.0
89,Subaru,2004,210703.0,NaN,2300.0
128,Audi,2015,98527.0,NaN,9995.0
129,Lucid,2023,1300.0,NaN,86900.0
131,Audi,2012,72922.0,NaN,25999.0
139,Porsche,2013,18494.0,NaN,86900.0
224,Maserati,2015,28700.0,NaN,24900.0
235,BMW,2016,45800.0,NaN,36800.0
357,Mercedes-Benz,2019,16813.0,NaN,45900.0
376,Hyundai,2022,26895.0,NaN,29800.0


In [38]:
df_clean[df_clean["accident"].isna()]["brand"].value_counts().head(20)

brand
Porsche          15
Ford             14
BMW               8
Chevrolet         7
Audi              6
Mercedes-Benz     5
Jeep              5
Hyundai           4
Land              4
GMC               3
INFINITI          3
Toyota            3
Cadillac          3
Honda             3
Alfa              3
Rivian            2
Bentley           2
Lexus             2
Lincoln           2
Nissan            2
Name: count, dtype: int64

In [39]:
df_clean[df_clean["accident"].isna()]["price"].describe()

count       113.000000
mean      50788.389381
std       45260.414855
min        2300.000000
25%       22450.000000
50%       36500.000000
75%       60000.000000
max      244896.000000
Name: price, dtype: float64

the missing accident values do not show an obvious pattern
We have missing accident information across:

 - luxury and non-luxury brands
 - old and newer vehicles
 - low and high mileage
 - low and high prices

### best approach is to create an 'unknown' category

In [40]:
df_clean['accident'] = df_clean['accident'].fillna('Unknown')

In [41]:
df_clean['accident'].value_counts()

accident
None reported                             2910
At least 1 accident or damage reported     986
Unknown                                    113
Name: count, dtype: int64

### now we go to transmission

In [42]:
df_clean["transmission"].value_counts()

transmission
A/T                                  1037
8-Speed A/T                           406
Transmission w/Dual Shift Mode        398
6-Speed A/T                           362
6-Speed M/T                           248
                                     ... 
7-Speed DCT Automatic                   1
9-Speed Automatic with Auto-Shift       1
SCHEDULED FOR OR IN PRODUCTION          1
6 Speed Mt                              1
8-Speed Manual                          1
Name: count, Length: 62, dtype: int64

In [43]:
#check manual transmission cars
df_clean["transmission"].str.contains("M/T|Mt|Manual", case=False, na=False).value_counts()

transmission
False    3633
True      376
Name: count, dtype: int64

 - 376 values containing manual related words
 - 3,633 don't contain manual related words

In [44]:
#check automatic transmission cars
df_clean["transmission"].str.contains("A/T|Automatic", case=False, na=False).value_counts()

transmission
True     3148
False     861
Name: count, dtype: int64


 - 3,148 contain A/T or Automatic
 - 861 don't contain either 

now we check which have not been captured in those categories

In [45]:
df_clean.loc[
    ~df_clean["transmission"].str.contains("M/T|Mt|Manual|A/T|Automatic", case=False, na=False),
    "transmission"
].value_counts()

transmission
Transmission w/Dual Shift Mode    398
CVT Transmission                   62
Transmission Overdrive Switch       7
Variable                            4
–                                   4
2                                   3
F                                   2
CVT-F                               1
8-SPEED AT                          1
Auto, 6-Spd w/CmdShft               1
6-Speed                             1
Single-Speed Fixed Gear             1
7-Speed                             1
SCHEDULED FOR OR IN PRODUCTION      1
Name: count, dtype: int64

### now lets deal with transmission
 - create new default category 'other'
 - identify descriptions containing manual-related words and change those to manual
 - identify automatic related descriptions and change those to automatic
 

In [46]:
df_clean["transmission_type"] = "Other"

df_clean.loc[
    df_clean["transmission"].str.contains(
        "M/T|Mt|Manual",
        case=False,
        na=False
    ),
    "transmission_type"
] = "Manual"

df_clean.loc[
    df_clean["transmission"].str.contains(
        "A/T|Automatic|Auto|CVT|Variable|Dual Shift|Overdrive|Fixed Gear",
        case=False,
        na=False
    ),
    "transmission_type"
] = "Automatic"

In [47]:
df_clean["transmission_type"].value_counts()

transmission_type
Automatic    3622
Manual        374
Other          13
Name: count, dtype: int64

'other' cassifies categories i could not from their description

In [48]:
#check the category other under transmission_type
df_clean.loc[df_clean["transmission_type"] == "Other", "transmission"]

5                                    F
165                         8-SPEED AT
269                                  2
476                                  F
516                                  2
536                                  –
855                                  –
916                                  –
1236                           6-Speed
1356                           7-Speed
1615                                 –
2381                                 2
2620    SCHEDULED FOR OR IN PRODUCTION
Name: transmission, dtype: str

In [49]:
pd.crosstab(
    df_clean["transmission_type"],
    df_clean["transmission"]
).T

transmission_type,Automatic,Manual,Other
transmission,,,
1-Speed A/T,64,0,0
1-Speed Automatic,14,0,0
10-Speed A/T,119,0,0
10-Speed Automatic,56,0,0
10-Speed Automatic with Overdrive,1,0,0
...,...,...,...
Single-Speed Fixed Gear,1,0,0
Transmission Overdrive Switch,7,0,0
Transmission w/Dual Shift Mode,398,0,0


### engine details extraction

engine has 1146 unique descriptons
 - we could extract engine size, hosepower and number of cylinders

In [50]:
df_clean["engine"].head(20)

0     300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...
1                                  3.8L V6 24V GDI DOHC
2                                        3.5 Liter DOHC
3     354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...
4                            2.0L I4 16V GDI DOHC Turbo
5                                             2.4 Liter
6          292.0HP 2.0L 4 Cylinder Engine Gasoline Fuel
7          282.0HP 4.4L 8 Cylinder Engine Gasoline Fuel
8         311.0HP 3.5L V6 Cylinder Engine Gasoline Fuel
9           534.0HP Electric Motor Electric Fuel System
10                                                   V6
11        715.0HP 5.2L 12 Cylinder Engine Gasoline Fuel
12    382.0HP 3.0L Straight 6 Cylinder Engine Gasoli...
13        400.0HP 3.0L V6 Cylinder Engine Gasoline Fuel
14                               2.0 Liter Supercharged
15         375.0HP 5.0L 8 Cylinder Engine Gasoline Fuel
16                           2.0L I4 16V GDI DOHC Turbo
17        305.0HP 3.6L V6 Cylinder Engine Gasoli

In [51]:
df_clean["engine"].value_counts().head(30)

engine
2.0L I4 16V GDI DOHC Turbo                               52
355.0HP 5.3L 8 Cylinder Engine Gasoline Fuel             48
420.0HP 6.2L 8 Cylinder Engine Gasoline Fuel             47
–                                                        45
300.0HP 3.0L Straight 6 Cylinder Engine Gasoline Fuel    44
240.0HP 2.0L 4 Cylinder Engine Gasoline Fuel             42
285.0HP 3.6L V6 Cylinder Engine Gasoline Fuel            40
5.7L V8 16V MPFI OHV                                     29
340.0HP 3.0L V6 Cylinder Engine Gasoline Fuel            28
3.6L V6 24V MPFI DOHC                                    28
3.6L V6 24V GDI DOHC                                     28
268.0HP 3.5L V6 Cylinder Engine Gasoline Fuel            24
455.0HP 6.2L 8 Cylinder Engine Gasoline Fuel             24
302.0HP 3.5L V6 Cylinder Engine Gasoline Fuel            23
490.0HP 6.2L 8 Cylinder Engine Gasoline Fuel             22
445.0HP 4.4L 8 Cylinder Engine Gasoline Fuel             21
295.0HP 3.5L V6 Cylinder Engine G

In [52]:
#check electric_engine cars
df_clean["engine"].str.contains(
    "Electric",
    case=False,
    na=False
).value_counts()

engine
False    3662
True      347
Name: count, dtype: int64

#### extract engine size

In [53]:
df_clean["engine_size"] = (
    df_clean["engine"]
    .str.extract(r"(\d+(?:\.\d+)?)\s*[Ll]")
    .astype(float)
)

In [54]:
df_clean[["engine", "engine_size"]].head(20)

,engine,engine_size
0,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,3.7
1,3.8L V6 24V GDI DOHC,3.8
2,3.5 Liter DOHC,3.5
3,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,3.5
4,2.0L I4 16V GDI DOHC Turbo,2.0
5,2.4 Liter,2.4
6,292.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,2.0
7,282.0HP 4.4L 8 Cylinder Engine Gasoline Fuel,4.4
8,311.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,3.5
9,534.0HP Electric Motor Electric Fuel System,NaN


In [55]:
df_clean["engine_size"].isnull().sum()

np.int64(217)

see all missing engine descriptions

In [56]:
df_clean.loc[
    df_clean["engine_size"].isna(),
    "engine"
].value_counts().head(30)

engine
–                                               45
Electric                                        18
835.0HP Electric Motor Electric Fuel System     16
425.0HP Electric Motor Electric Fuel System     16
455.0HP Electric Motor Electric Fuel System     12
518.0HP Electric Motor Electric Fuel System     11
271.0HP Electric Motor Electric Fuel System      9
Electric Motor Electric Fuel System              9
670.0HP Electric Motor Electric Fuel System      7
200.0HP Electric Motor Electric Fuel System      4
355.0HP Electric Motor Electric Fuel System      3
329.0HP Electric Motor Electric Fuel System      3
Dual Motor - Standard                            3
201.0HP Electric Motor Electric Fuel System      3
320.0HP Electric Motor Electric Fuel System      3
295.0HP Electric Motor Electric Fuel System      3
480.0HP Electric Motor Electric Fuel System      3
534.0HP Electric Motor Electric Fuel System      2
V6                                               2
536.0HP Electric Motor E

check how many are electric

In [57]:
df_clean.loc[
    df_clean["engine_size"].isna(),
    "engine"
].str.contains("Electric", case=False, na=False).value_counts()

engine
True     161
False     56
Name: count, dtype: int64

217 missing engine size
 - 161 electric-related
 - 56 unknown

create is_electric where 
 - electric - 1
 - not electric -0

In [58]:
df_clean["is_electric"] = (
    df_clean["engine"]
    .str.contains("Electric", case=False, na=False)
    .astype(int)
)

In [59]:
df_clean["is_electric"].value_counts()

is_electric
0    3662
1     347
Name: count, dtype: int64

### extract horsepower

In [60]:
df_clean["horsepower"] = (
    df_clean["engine"]
    .str.extract(r"(\d+(?:\.\d+)?)\s*HP")
    .astype(float)
)

In [61]:
df_clean[["engine", "horsepower"]].head(20)

,engine,horsepower
0,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,300.0
1,3.8L V6 24V GDI DOHC,NaN
2,3.5 Liter DOHC,NaN
3,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,354.0
4,2.0L I4 16V GDI DOHC Turbo,NaN
5,2.4 Liter,NaN
6,292.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,292.0
7,282.0HP 4.4L 8 Cylinder Engine Gasoline Fuel,282.0
8,311.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,311.0
9,534.0HP Electric Motor Electric Fuel System,534.0


In [62]:
df_clean["horsepower"].isna().sum()

np.int64(808)

In [63]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   brand              4009 non-null   str    
 1   model_year         4009 non-null   int64  
 2   milage             4009 non-null   float64
 3   fuel_type          4009 non-null   str    
 4   engine             4009 non-null   str    
 5   transmission       4009 non-null   str    
 6   ext_col            4009 non-null   str    
 7   int_col            4009 non-null   str    
 8   accident           4009 non-null   str    
 9   price              4009 non-null   float64
 10  transmission_type  4009 non-null   str    
 11  engine_size        3792 non-null   float64
 12  is_electric        4009 non-null   int64  
 13  horsepower         3201 non-null   float64
dtypes: float64(4), int64(2), str(8)
memory usage: 438.6 KB


In [64]:
#check columns with missing values
df_clean.isnull().sum()

brand                  0
model_year             0
milage                 0
fuel_type              0
engine                 0
transmission           0
ext_col                0
int_col                0
accident               0
price                  0
transmission_type      0
engine_size          217
is_electric            0
horsepower           808
dtype: int64

In [65]:
df_clean[["engine", "engine_size", "horsepower", "is_electric"]].head(20)

,engine,engine_size,horsepower,is_electric
0,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,3.7,300.0,0
1,3.8L V6 24V GDI DOHC,3.8,NaN,0
2,3.5 Liter DOHC,3.5,NaN,0
3,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,3.5,354.0,1
4,2.0L I4 16V GDI DOHC Turbo,2.0,NaN,0
5,2.4 Liter,2.4,NaN,0
6,292.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,2.0,292.0,0
7,282.0HP 4.4L 8 Cylinder Engine Gasoline Fuel,4.4,282.0,0
8,311.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,3.5,311.0,0
9,534.0HP Electric Motor Electric Fuel System,NaN,534.0,1


In [66]:
df_clean[["engine_size", "horsepower", "is_electric"]].isna().sum()

engine_size    217
horsepower     808
is_electric      0
dtype: int64

drop 'engine' column because we have the information we need

In [67]:
df_clean = df_clean.drop(columns=["engine"])

In [68]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   brand              4009 non-null   str    
 1   model_year         4009 non-null   int64  
 2   milage             4009 non-null   float64
 3   fuel_type          4009 non-null   str    
 4   transmission       4009 non-null   str    
 5   ext_col            4009 non-null   str    
 6   int_col            4009 non-null   str    
 7   accident           4009 non-null   str    
 8   price              4009 non-null   float64
 9   transmission_type  4009 non-null   str    
 10  engine_size        3792 non-null   float64
 11  is_electric        4009 non-null   int64  
 12  horsepower         3201 non-null   float64
dtypes: float64(4), int64(2), str(7)
memory usage: 407.3 KB


### create car_age column
use 2024 because the dataset is from 2024 to get the age of the cars at that time and their prices

In [69]:
df_clean["car_age"] = 2024 - df_clean["model_year"]

In [70]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   brand              4009 non-null   str    
 1   model_year         4009 non-null   int64  
 2   milage             4009 non-null   float64
 3   fuel_type          4009 non-null   str    
 4   transmission       4009 non-null   str    
 5   ext_col            4009 non-null   str    
 6   int_col            4009 non-null   str    
 7   accident           4009 non-null   str    
 8   price              4009 non-null   float64
 9   transmission_type  4009 non-null   str    
 10  engine_size        3792 non-null   float64
 11  is_electric        4009 non-null   int64  
 12  horsepower         3201 non-null   float64
 13  car_age            4009 non-null   int64  
dtypes: float64(4), int64(3), str(7)
memory usage: 438.6 KB
